# 04b - Binary Model Ensemble

Train separate binary classifiers for each outcome instead of a single 7-class model.

**Motivation:**
- The multiclass model spreads importance across 207 features
- K-rate features only get 5% of total importance
- Separate models allow outcome-specific feature selection
- Better calibration for extreme pitchers (Skubal, Skenes)

**Approach:**
- Train 7 binary classifiers: K vs not-K, BB vs not-BB, etc.
- Feature selection per model using importance thresholding
- Normalize outputs to valid probability distribution
- Compare to multiclass model

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle
import shap

from src.model.train_binary_models import (
    BinaryModelEnsemble, 
    OUTCOME_CLASSES, 
    compare_with_multiclass,
    prepare_features,
)

# Paths
data_dir = Path('../data/processed')
model_dir = Path('../models')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

/Applications/miniconda3/envs/mlbenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Memory management utilities
import gc
import psutil

def clear_mem():
    """Run garbage collection and print memory usage."""
    gc.collect()
    mem_gb = psutil.Process().memory_info().rss / 1024**3
    print(f"Memory: {mem_gb:.2f} GB")
    return mem_gb

clear_mem()

## 1. Load and Prepare Data

In [2]:
# Load the matchup data
matchups = pd.read_parquet(data_dir / 'matchups.parquet')
print(f"Loaded {len(matchups):,} matchups")
print(f"Seasons: {sorted(matchups['season'].unique())}")
print(f"Outcome distribution:")
print(matchups['outcome'].value_counts(normalize=True).round(3))

clear_mem()

Loaded 987,069 matchups
Seasons: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
Outcome distribution:
outcome
OUT    0.464
K      0.231
1B     0.143
BB     0.083
2B     0.044
HR     0.031
3B     0.004
Name: proportion, dtype: float64


In [3]:
# Split by season
seasons = matchups['season'].values
unique_seasons = sorted(matchups['season'].unique())
print(f"Available seasons: {unique_seasons}")

# Determine splits based on available data
if 2026 in unique_seasons:
    train_mask = seasons <= 2024
    val_mask = seasons == 2025
    test_mask = seasons == 2026
else:
    train_mask = seasons < 2024
    val_mask = seasons == 2024
    test_mask = seasons == 2025

train_df = matchups[train_mask]
val_df = matchups[val_mask]
test_df = matchups[test_mask]

print(f"\nTrain: {len(train_df):,} samples")
print(f"Val: {len(val_df):,} samples")
print(f"Test: {len(test_df):,} samples")

Available seasons: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

Train: 777,702 samples
Val: 195,057 samples
Test: 14,310 samples


In [4]:
# Prepare features using the preprocessor
preprocessor_path = str(model_dir / 'matchup_preprocessor.pkl')

X_train, y_train, feature_names, outcome_classes = prepare_features(train_df, preprocessor_path)
X_val, y_val, _, _ = prepare_features(val_df, preprocessor_path)
X_test, y_test, _, _ = prepare_features(test_df, preprocessor_path)

print(f"Feature count: {len(feature_names)}")
print(f"Outcome classes: {outcome_classes}")
print(f"\nTrain shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

clear_mem()

Feature count: 207
Outcome classes: [np.str_('1B'), np.str_('2B'), np.str_('3B'), np.str_('BB'), np.str_('HR'), np.str_('K'), np.str_('OUT')]

Train shape: (777702, 207)
Val shape: (195057, 207)
Test shape: (14310, 207)


## 2. Train Binary Model Ensemble

Settings:
- Time budget: 180s per model (7 models = ~21 min total)
- Feature selection enabled (keeps features with importance > 1% of max)
- min_num_leaves = 32 to avoid regression to mean

In [5]:
# Configuration
TIME_BUDGET_PER_MODEL = 60  # 3 min per model
MIN_NUM_LEAVES = 16
FEATURE_SELECTION = True
FEATURE_SELECTION_THRESHOLD = 0.20 # Keep features with >= X% of max importance

In [6]:
# Train ensemble
ensemble = BinaryModelEnsemble(
    time_budget_per_model=TIME_BUDGET_PER_MODEL,
    metric='log_loss',
    min_num_leaves=MIN_NUM_LEAVES,
    feature_selection=FEATURE_SELECTION,
    feature_selection_threshold=FEATURE_SELECTION_THRESHOLD,
)

ensemble.fit(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    verbose=1,
    save_dir = model_dir / 'binary_ensemble',
)

clear_mem()

Training 7 binary classifiers
  Time budget per model: 60s
  Feature selection: True
  Memory efficient: True
  Train shape: (777702, 207)

[1/7] Training 1B vs not-1B...
  Positive rate: 14.3% (111,135 / 777,702)
  Selected features: 84 / 207
  Best model: catboost
  Best log_loss: 0.4119
  Val AUC: 0.553, Brier: 0.1234
  Saved to ../models/binary_ensemble/model_1B.pkl

[2/7] Training 2B vs not-2B...
  Positive rate: 4.4% (34,513 / 777,702)
  Selected features: 151 / 207
  Best model: lgbm
  Best log_loss: 0.1777
  Val AUC: 0.501, Brier: 0.0412
  Saved to ../models/binary_ensemble/model_2B.pkl

[3/7] Training 3B vs not-3B...
  Positive rate: 0.4% (2,960 / 777,702)
  Selected features: 148 / 207
  Best model: lgbm
  Best log_loss: 0.0235
  Val AUC: 0.514, Brier: 0.0035
  Saved to ../models/binary_ensemble/model_3B.pkl

[4/7] Training BB vs not-BB...
  Positive rate: 8.3% (64,373 / 777,702)
  Selected features: 45 / 207
  Best model: catboost
  Best log_loss: 0.2807
  Val AUC: 0.605, Br

In [7]:
# Summary of all models
summary = ensemble.summary()
summary

""


## 3. Feature Selection Results

See which features were selected for each outcome model.

In [8]:
# Features selected per model
print("Features selected per outcome model:")
print("=" * 50)
for outcome in OUTCOME_CLASSES:
    features = ensemble.selected_features.get(outcome, [])
    print(f"\n{outcome}: {len(features)} features")
    
# Compare overlap
k_features = set(ensemble.selected_features.get('K', []))
hr_features = set(ensemble.selected_features.get('HR', []))
bb_features = set(ensemble.selected_features.get('BB', []))

print(f"\n\nFeature overlap:")
print(f"K ∩ HR: {len(k_features & hr_features)} features")
print(f"K ∩ BB: {len(k_features & bb_features)} features")
print(f"HR ∩ BB: {len(hr_features & bb_features)} features")
print(f"All three: {len(k_features & hr_features & bb_features)} features")

Features selected per outcome model:

1B: 84 features

2B: 151 features

3B: 148 features

BB: 45 features

HR: 78 features

K: 35 features

OUT: 58 features


Feature overlap:
K ∩ HR: 19 features
K ∩ BB: 10 features
HR ∩ BB: 31 features
All three: 5 features


In [9]:
# K-specific features (in K model but not HR model)
k_specific = k_features - hr_features
print(f"K-specific features ({len(k_specific)}):")
for f in sorted(k_specific):
    if 'k_rate' in f.lower() or 'whiff' in f.lower() or 'contact' in f.lower():
        print(f"  {f}")

K-specific features (16):
  b_roll100_k_rate
  b_velo_94_96_whiff_pct
  b_velo_97_plus_whiff_pct
  b_vs_LHP_whiff_pct
  b_vs_RHP_whiff_pct
  b_vs_ff_whiff_pct
  b_vs_sl_whiff_pct
  b_whiff_pct
  p_contact_pct
  p_ff_whiff_pct
  p_roll10_whiff_rate
  p_whiff_pct


In [10]:
# HR-specific features (in HR model but not K model)
hr_specific = hr_features - k_features
print(f"HR-specific features ({len(hr_specific)}):")
for f in sorted(hr_specific):
    if 'barrel' in f.lower() or 'exit' in f.lower() or 'launch' in f.lower() or 'hr' in f.lower():
        print(f"  {f}")

HR-specific features (59):
  b_avg_exit_velo
  b_barrel_pct
  b_max_exit_velo
  b_roll100_avg_exit_velo
  b_roll100_barrel_rate
  b_roll25_avg_exit_velo
  b_roll25_barrel_rate
  b_roll50_avg_exit_velo
  b_roll50_barrel_rate
  p_roll10_avg_exit_velo
  p_roll10_barrel_rate
  p_roll20_avg_exit_velo
  p_roll5_avg_exit_velo


## 4. Feature Importance via SHAP

Examine what drives each outcome model.

In [11]:
# Sample for SHAP (use subset for speed)
shap_sample_size = min(1000, len(X_test))
X_shap = X_test.sample(shap_sample_size, random_state=42)

In [12]:
# K Model SHAP
print("K Model - SHAP Summary")
fig = ensemble.plot_shap_summary(X_shap, 'K', max_display=20)
plt.show()

K Model - SHAP Summary


KeyError: 'K'

In [ ]:
# BB Model SHAP
print("BB Model - SHAP Summary")
fig = ensemble.plot_shap_summary(X_shap, 'BB', max_display=20)
plt.show()

In [ ]:
# HR Model SHAP
print("HR Model - SHAP Summary")
fig = ensemble.plot_shap_summary(X_shap, 'HR', max_display=20)
plt.show()

In [ ]:
# OUT Model SHAP
print("OUT Model - SHAP Summary")
fig = ensemble.plot_shap_summary(X_shap, 'OUT', max_display=20)
plt.show()

## 5. Feature Importance Comparison (Tree-based)

Compare top features across models.

In [ ]:
# Get feature importance DataFrames
importance_dfs = {}
for outcome in OUTCOME_CLASSES:
    importance_dfs[outcome] = ensemble.get_feature_importance_df(outcome)

# Show top 15 for K, BB, HR
fig, axes = plt.subplots(1, 3, figsize=(15, 8))

for ax, outcome in zip(axes, ['K', 'BB', 'HR']):
    df = importance_dfs[outcome].head(15)
    ax.barh(df['feature'], df['importance'])
    ax.set_xlabel('Importance')
    ax.set_title(f'{outcome} Model - Top 15 Features')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [13]:
# K model - check K-rate feature importance specifically
k_imp = importance_dfs['K']
print("K Model - K-rate related features:")
k_rate_features = k_imp[k_imp['feature'].str.contains('k_rate|whiff|contact|chase', case=False)]
print(k_rate_features.head(20))

total_imp = k_imp['importance'].sum()
k_rate_imp = k_rate_features['importance'].sum()
print(f"\nK-rate features: {100*k_rate_imp/total_imp:.1f}% of total importance")
print(f"(vs 5.1% in multiclass model)")

NameError: name 'importance_dfs' is not defined

## 6. Compare to Multiclass Model

In [14]:
# Load multiclass model for comparison
try:
    with open(model_dir / 'flaml_trainer.pkl', 'rb') as f:
        multiclass_trainer = pickle.load(f)
    
    # Compare predictions
    comparison = compare_with_multiclass(
        binary_ensemble=ensemble,
        multiclass_trainer=multiclass_trainer,
        X_test=X_test,
        y_test=y_test,
    )
    
    print("Binary vs Multiclass Comparison:")
    print(comparison.to_string(index=False))
    
    print(f"\nAverage AUC improvement: {comparison['auc_diff'].mean():.4f}")
    print(f"Average Brier improvement: {comparison['brier_diff'].mean():.4f}")
    
except FileNotFoundError:
    print("Multiclass model not found - skipping comparison")

Binary vs Multiclass Comparison:
outcome  binary_auc  multi_auc  auc_diff  binary_brier  multi_brier  brier_diff
     1B    0.540474   0.535526  0.004948      0.119832     0.120011    0.000179
     2B    0.507995   0.543647 -0.035653      0.042345     0.042303   -0.000042
     3B    0.487222   0.550231 -0.063008      0.004107     0.004110    0.000003
     BB    0.594329   0.603191 -0.008862      0.086896     0.086715   -0.000181
     HR    0.608651   0.607661  0.000989      0.030822     0.030808   -0.000015
      K    0.589583   0.589059  0.000524      0.178924     0.179410    0.000486
    OUT    0.562165   0.558289  0.003875      0.243936     0.244424    0.000488

Average AUC improvement: -0.0139
Average Brier improvement: 0.0001


## 7. Test on Elite Pitchers

Check predictions for Skubal, Skenes, etc.

In [15]:
# Load pitcher rolling stats
pitcher_rolling = pd.read_csv('../data/profiles/pitcher_rolling.csv')
pitcher_arsenal = pd.read_csv('../data/profiles/pitcher_arsenal.csv')
batter_rolling = pd.read_csv('../data/profiles/batter_rolling.csv')

# Find elite K pitchers
elite_pitchers = pitcher_rolling[pitcher_rolling['p_roll10_k_rate'] > 0.30]
elite_ids = elite_pitchers['pitcher_id'].tolist()

# Get names
elite_names = pitcher_arsenal[pitcher_arsenal['pitcher_id'].isin(elite_ids)][['pitcher_id', 'pitcher_name']].drop_duplicates()
print(f"Elite K pitchers (>30% K rate): {len(elite_names)}")
print(elite_names.head(10))

clear_mem()

Elite K pitchers (>30% K rate): 183
     pitcher_id      pitcher_name
9        445276    Jansen, Kenley
14       448281   Doolittle, Sean
42       465657     Soria, Joakim
59       493603    Ottavino, Adam
66       501789      Harris, Will
78       502327  Santiago, Héctor
84       503285     O'Day, Darren
95       518585    Cruz, Fernando
100      518813     Holland, Greg
103      518886    Kimbrel, Craig


In [16]:
# Find avg K rate batters for testing
avg_batters = batter_rolling[
    (batter_rolling['b_roll25_k_rate'] >= 0.22) & 
    (batter_rolling['b_roll25_k_rate'] <= 0.24)
]
print(f"Average K rate batters: {len(avg_batters)}")

Average K rate batters: 133


## 8. Save Ensemble

In [17]:
# Save ensemble
ensemble.save(model_dir / 'binary_ensemble.pkl')

# Save feature importance CSVs
for outcome in OUTCOME_CLASSES:
    df = ensemble.get_feature_importance_df(outcome)
    df.to_csv(model_dir / f'feature_importance_{outcome}.csv', index=False)

print(f"Saved ensemble and feature importance files to {model_dir}")

clear_mem()

AttributeError: 'BinaryModelEnsemble' object has no attribute 'save'

In [ ]:
# Final summary
print("\n" + "=" * 60)
print("BINARY ENSEMBLE TRAINING COMPLETE")
print("=" * 60)
print(f"\nModels trained: {len(ensemble.models)}")
print(f"\nFeature counts per model:")
for outcome in OUTCOME_CLASSES:
    n_feat = len(ensemble.selected_features.get(outcome, []))
    print(f"  {outcome}: {n_feat} features")

print(f"\nValidation metrics:")
for outcome, metrics in ensemble.metrics.items():
    print(f"  {outcome}: AUC={metrics['auc']:.3f}, Brier={metrics['brier']:.4f}")